# Standalone DOCX to PDF Converter for NAS Drive

This notebook converts all DOCX files to PDF on a NAS drive, searching recursively through all folders.

**Features:**
- Connects to NAS using the same method as iris-project-database-refresh scripts
- Recursively searches all folders for DOCX files
- Converts using LibreOffice, Pandoc, or Python (tries in that order)
- Saves PDFs in the same location as original DOCX files
- Parallel processing for faster conversion
- Skips existing PDFs to avoid re-conversion

**Requirements:**
- For best results, install LibreOffice: `brew install --cask libreoffice`
- Or install Pandoc: `brew install pandoc`
- Python libraries will be installed automatically if needed

## Step 1: Install Required Libraries

In [ ]:
# Install required Python libraries
import subprocess
import sys

def install_package(package):
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed")

# Required packages
packages = [
    "smbclient",
    "python-docx", 
    "reportlab"
]

for package in packages:
    install_package(package)

print("\nAll required packages are installed!")

## Step 2: Configuration - Update Your NAS Details

In [ ]:
# ========================================
# UPDATE THESE VALUES WITH YOUR NAS DETAILS
# ========================================

NAS_CONFIG = {
    "ip": "your_nas_ip_address",     # e.g., "192.168.1.100"
    "share": "your_share_name",      # e.g., "documents" 
    "user": "your_nas_username",
    "password": "your_nas_password"
}

# Path on NAS to search (leave empty "" to search entire share)
SEARCH_PATH = ""  # e.g., "contracts/2024" or "" for entire share

# Conversion settings
MAX_WORKERS = 3        # Number of parallel conversion workers
SKIP_EXISTING = True   # Skip conversion if PDF already exists

print(f"Configuration set for NAS: {NAS_CONFIG['ip']}/{NAS_CONFIG['share']}")
print(f"Search path: {'Entire share' if not SEARCH_PATH else SEARCH_PATH}")
print(f"Parallel workers: {MAX_WORKERS}")
print(f"Skip existing PDFs: {SKIP_EXISTING}")

## Step 3: Import Libraries and Define Functions

In [ ]:
import os
import logging
import subprocess
import tempfile
from pathlib import Path
from typing import List, Optional, Tuple
import smbclient
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# Configure logging for the notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)

print("Libraries imported successfully!")

In [ ]:
# Conversion method checkers
def check_libreoffice_available() -> bool:
    """Check if LibreOffice is available on the system."""
    try:
        result = subprocess.run(['libreoffice', '--version'], 
                              capture_output=True, text=True, timeout=10)
        return result.returncode == 0
    except (subprocess.TimeoutExpired, FileNotFoundError):
        return False

def check_pandoc_available() -> bool:
    """Check if Pandoc is available on the system."""
    try:
        result = subprocess.run(['pandoc', '--version'], 
                              capture_output=True, text=True, timeout=10)
        return result.returncode == 0
    except (subprocess.TimeoutExpired, FileNotFoundError):
        return False

# Check available methods
libreoffice_available = check_libreoffice_available()
pandoc_available = check_pandoc_available()

print(f"LibreOffice available: {libreoffice_available}")
print(f"Pandoc available: {pandoc_available}")
print("Python conversion (fallback): Always available")

if not libreoffice_available and not pandoc_available:
    print("\n⚠️  For best results, install LibreOffice or Pandoc:")
    print("   brew install --cask libreoffice")
    print("   brew install pandoc")

In [ ]:
# Conversion functions
def convert_docx_to_pdf_libreoffice(docx_path: str, output_dir: str) -> bool:
    """Convert DOCX to PDF using LibreOffice command line."""
    try:
        cmd = [
            'libreoffice',
            '--headless',
            '--convert-to', 'pdf',
            '--outdir', output_dir,
            docx_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        
        if result.returncode == 0:
            logger.info(f"LibreOffice conversion successful: {os.path.basename(docx_path)}")
            return True
        else:
            logger.error(f"LibreOffice conversion failed: {result.stderr}")
            return False
            
    except subprocess.TimeoutExpired:
        logger.error(f"LibreOffice conversion timeout: {os.path.basename(docx_path)}")
        return False
    except Exception as e:
        logger.error(f"LibreOffice conversion error: {e}")
        return False

def convert_docx_to_pdf_pandoc(docx_path: str, pdf_path: str) -> bool:
    """Convert DOCX to PDF using Pandoc."""
    try:
        cmd = [
            'pandoc',
            docx_path,
            '-o', pdf_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        
        if result.returncode == 0:
            logger.info(f"Pandoc conversion successful: {os.path.basename(docx_path)}")
            return True
        else:
            logger.error(f"Pandoc conversion failed: {result.stderr}")
            return False
            
    except subprocess.TimeoutExpired:
        logger.error(f"Pandoc conversion timeout: {os.path.basename(docx_path)}")
        return False
    except Exception as e:
        logger.error(f"Pandoc conversion error: {e}")
        return False

def convert_docx_to_pdf_python(docx_path: str, pdf_path: str) -> bool:
    """Convert DOCX to PDF using pure Python libraries (fallback method)."""
    try:
        from docx import Document
        from reportlab.lib.pagesizes import letter
        from reportlab.platypus import SimpleDocTemplate, Paragraph
        from reportlab.lib.styles import getSampleStyleSheet
        
        # Read DOCX content
        doc = Document(docx_path)
        text_content = []
        
        for paragraph in doc.paragraphs:
            if paragraph.text.strip():
                text_content.append(paragraph.text)
        
        # Create PDF
        pdf_doc = SimpleDocTemplate(pdf_path, pagesize=letter)
        styles = getSampleStyleSheet()
        story = []
        
        for text in text_content:
            # Clean text for reportlab
            clean_text = text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
            para = Paragraph(clean_text, styles['Normal'])
            story.append(para)
        
        pdf_doc.build(story)
        logger.info(f"Python conversion successful: {os.path.basename(docx_path)}")
        return True
        
    except ImportError as e:
        logger.warning(f"Python conversion libraries not available: {e}")
        return False
    except Exception as e:
        logger.error(f"Python conversion error: {e}")
        return False

print("Conversion functions defined!")

In [ ]:
# NAS operations
def initialize_smb_client() -> bool:
    """Sets up smbclient credentials."""
    try:
        smbclient.ClientConfig(username=NAS_CONFIG["user"], password=NAS_CONFIG["password"])
        logger.info("SMB client configured successfully.")
        return True
    except Exception as e:
        logger.error(f"Failed to configure SMB client: {e}")
        return False

def find_docx_files_recursive(base_smb_path: str) -> List[str]:
    """Recursively find all DOCX files in the specified NAS path and all subfolders."""
    docx_files = []
    
    def scan_directory(current_path: str, relative_path: str = ""):
        try:
            for item in smbclient.listdir(current_path):
                item_path = os.path.join(current_path, item).replace('\\', '/')
                item_relative_path = os.path.join(relative_path, item).replace('\\', '/') if relative_path else item
                
                try:
                    if smbclient.path.isfile(item_path):
                        if item.lower().endswith('.docx') and not item.startswith('~'):
                            docx_files.append(item_relative_path)
                            print(f"Found: {item_relative_path}")
                    elif smbclient.path.isdir(item_path):
                        # Recursively scan subdirectory
                        scan_directory(item_path, item_relative_path)
                except Exception as e:
                    logger.warning(f"Error processing item '{item}': {e}")
                    
        except Exception as e:
            logger.error(f"Error scanning directory '{current_path}': {e}")
    
    logger.info(f"Scanning for DOCX files in: {base_smb_path}")
    scan_directory(base_smb_path)
    logger.info(f"Found {len(docx_files)} DOCX files")
    
    return docx_files

def download_file_from_nas(smb_file_path: str, local_file_path: str) -> bool:
    """Download a file from NAS to local filesystem."""
    try:
        with smbclient.open_file(smb_file_path, mode='rb') as f_remote:
            with open(local_file_path, 'wb') as f_local:
                f_local.write(f_remote.read())
        return True
    except Exception as e:
        logger.error(f"Error downloading file '{smb_file_path}': {e}")
        return False

def upload_file_to_nas(local_file_path: str, smb_file_path: str) -> bool:
    """Upload a file from local filesystem to NAS."""
    try:
        # Ensure the directory exists on NAS
        smb_dir = os.path.dirname(smb_file_path)
        if not smbclient.path.exists(smb_dir):
            smbclient.makedirs(smb_dir, exist_ok=True)
            
        with open(local_file_path, 'rb') as f_local:
            with smbclient.open_file(smb_file_path, mode='wb') as f_remote:
                f_remote.write(f_local.read())
        return True
    except Exception as e:
        logger.error(f"Error uploading file '{smb_file_path}': {e}")
        return False

def pdf_exists_on_nas(smb_pdf_path: str) -> bool:
    """Check if PDF file already exists on NAS."""
    try:
        return smbclient.path.exists(smb_pdf_path)
    except Exception:
        return False

print("NAS operation functions defined!")

In [ ]:
# Main conversion function
def convert_single_docx_file(docx_relative_path: str, base_smb_path: str, 
                           conversion_methods: List[str], skip_existing: bool = True) -> Tuple[bool, str]:
    """Convert a single DOCX file to PDF."""
    # Construct full SMB paths
    smb_docx_path = os.path.join(base_smb_path, docx_relative_path).replace('\\', '/')
    
    # Determine PDF path (same location as DOCX, with .pdf extension)
    pdf_relative_path = os.path.splitext(docx_relative_path)[0] + '.pdf'
    smb_pdf_path = os.path.join(base_smb_path, pdf_relative_path).replace('\\', '/')
    
    # Check if PDF already exists
    if skip_existing and pdf_exists_on_nas(smb_pdf_path):
        logger.info(f"PDF already exists, skipping: {pdf_relative_path}")
        return True, "PDF already exists"
    
    logger.info(f"Converting: {docx_relative_path} -> {pdf_relative_path}")
    
    # Create temporary directory for processing
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_docx_path = os.path.join(temp_dir, 'input.docx')
        temp_pdf_path = os.path.join(temp_dir, 'output.pdf')
        
        # Download DOCX file
        if not download_file_from_nas(smb_docx_path, temp_docx_path):
            return False, "Failed to download DOCX file"
        
        # Try conversion methods in order
        conversion_successful = False
        last_error = ""
        
        for method in conversion_methods:
            try:
                if method == 'libreoffice':
                    if convert_docx_to_pdf_libreoffice(temp_docx_path, temp_dir):
                        # LibreOffice creates the PDF with the same name as input
                        expected_pdf = os.path.join(temp_dir, 'input.pdf')
                        if os.path.exists(expected_pdf):
                            os.rename(expected_pdf, temp_pdf_path)
                            conversion_successful = True
                            break
                elif method == 'pandoc':
                    if convert_docx_to_pdf_pandoc(temp_docx_path, temp_pdf_path):
                        conversion_successful = True
                        break
                elif method == 'python':
                    if convert_docx_to_pdf_python(temp_docx_path, temp_pdf_path):
                        conversion_successful = True
                        break
            except Exception as e:
                last_error = f"{method} failed: {e}"
                logger.warning(last_error)
        
        if not conversion_successful:
            return False, f"All conversion methods failed. Last error: {last_error}"
        
        # Upload PDF file back to NAS
        if not upload_file_to_nas(temp_pdf_path, smb_pdf_path):
            return False, "Failed to upload PDF file to NAS"
    
    logger.info(f"Successfully converted: {docx_relative_path}")
    return True, "Success"

print("Main conversion function defined!")

## Step 4: Test Connection and Find DOCX Files

In [ ]:
# Test NAS connection and find DOCX files
print("Testing NAS connection...")

if not initialize_smb_client():
    print("❌ Failed to initialize SMB client. Check your NAS configuration.")
else:
    print("✓ SMB client initialized successfully")
    
    # Construct base SMB path
    base_smb_path = f"//{NAS_CONFIG['ip']}/{NAS_CONFIG['share']}"
    if SEARCH_PATH:
        base_smb_path = os.path.join(base_smb_path, SEARCH_PATH).replace('\\', '/')
    
    print(f"Searching in: {base_smb_path}")
    
    # Check if path exists
    if not smbclient.path.exists(base_smb_path):
        print(f"❌ NAS path does not exist: {base_smb_path}")
    else:
        print("✓ NAS path exists")
        
        # Find DOCX files
        print("\nScanning for DOCX files...")
        docx_files = find_docx_files_recursive(base_smb_path)
        
        print(f"\n📊 Found {len(docx_files)} DOCX files")
        
        if docx_files:
            print("\nFirst 10 files:")
            for i, file_path in enumerate(docx_files[:10]):
                print(f"  {i+1:2d}. {file_path}")
            
            if len(docx_files) > 10:
                print(f"     ... and {len(docx_files) - 10} more files")
        else:
            print("No DOCX files found in the specified path.")

## Step 5: Run the Conversion

In [ ]:
# Main conversion process
def run_conversion_process():
    print("=" * 60)
    print("Starting DOCX to PDF Conversion")
    print("=" * 60)
    
    start_time = datetime.now()
    
    # Initialize SMB client
    if not initialize_smb_client():
        return {"error": "Failed to initialize SMB client"}
    
    # Construct base SMB path
    base_smb_path = f"//{NAS_CONFIG['ip']}/{NAS_CONFIG['share']}"
    if SEARCH_PATH:
        base_smb_path = os.path.join(base_smb_path, SEARCH_PATH).replace('\\', '/')
    
    print(f"Searching in: {base_smb_path}")
    
    # Check if path exists
    if not smbclient.path.exists(base_smb_path):
        error_msg = f"NAS path does not exist: {base_smb_path}"
        print(f"❌ {error_msg}")
        return {"error": error_msg}
    
    # Find all DOCX files
    docx_files = find_docx_files_recursive(base_smb_path)
    
    if not docx_files:
        print("No DOCX files found")
        return {"total_files": 0, "successful": 0, "failed": 0, "skipped": 0}
    
    # Determine available conversion methods
    available_methods = []
    if libreoffice_available:
        available_methods.append('libreoffice')
    if pandoc_available:
        available_methods.append('pandoc')
    available_methods.append('python')  # Always available as fallback
    
    print(f"Using conversion methods: {available_methods}")
    print(f"Starting conversion of {len(docx_files)} files with {MAX_WORKERS} workers")
    
    # Convert files in parallel
    results = {"total_files": len(docx_files), "successful": 0, "failed": 0, "skipped": 0}
    failed_files = []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_file = {
            executor.submit(convert_single_docx_file, docx_file, base_smb_path, 
                          available_methods, SKIP_EXISTING): docx_file 
            for docx_file in docx_files
        }
        
        for future in as_completed(future_to_file):
            docx_file = future_to_file[future]
            try:
                success, message = future.result()
                if success:
                    if "already exists" in message:
                        results["skipped"] += 1
                    else:
                        results["successful"] += 1
                else:
                    results["failed"] += 1
                    failed_files.append(f"{docx_file}: {message}")
                    print(f"❌ Failed: {docx_file} - {message}")
            except Exception as e:
                results["failed"] += 1
                failed_files.append(f"{docx_file}: {e}")
                print(f"❌ Exception: {docx_file} - {e}")
    
    end_time = datetime.now()
    duration = end_time - start_time
    
    # Summary
    print("\n" + "=" * 60)
    print("Conversion Summary")
    print("=" * 60)
    print(f"Duration: {duration}")
    print(f"Total files: {results['total_files']}")
    print(f"✓ Successful: {results['successful']}")
    print(f"⏭️  Skipped (already exist): {results['skipped']}")
    print(f"❌ Failed: {results['failed']}")
    
    if failed_files:
        print("\nFailed files:")
        for failed_file in failed_files[:10]:  # Show first 10 failures
            print(f"  - {failed_file}")
        if len(failed_files) > 10:
            print(f"  ... and {len(failed_files) - 10} more")
    
    results["failed_files"] = failed_files
    results["duration"] = str(duration)
    return results

# Run the conversion
conversion_results = run_conversion_process()
print(f"\nFinal Results: {conversion_results}")

## Optional: Re-run with Different Settings

You can modify the configuration above and re-run the conversion:

In [ ]:
# Example: Convert a specific folder
SEARCH_PATH = "contracts/2024"  # Change this to your target folder
SKIP_EXISTING = False  # Set to False to overwrite existing PDFs

print(f"Updated settings:")
print(f"Search path: {SEARCH_PATH}")
print(f"Skip existing: {SKIP_EXISTING}")

# Uncomment the line below to run conversion with new settings
# conversion_results = run_conversion_process()

## Troubleshooting

If you encounter issues:

1. **Connection errors**: Check NAS IP, share name, username, and password
2. **Permission errors**: Ensure your NAS user has read/write access to the files
3. **Conversion errors**: Install LibreOffice or Pandoc for better conversion quality
4. **File corruption**: Some DOCX files might be corrupted and cannot be converted
5. **Network timeouts**: Reduce MAX_WORKERS if you have a slow network connection

**Installation commands for better conversion:**
```bash
# Install LibreOffice (recommended)
brew install --cask libreoffice

# Install Pandoc (alternative)
brew install pandoc
```